In [5]:
import os
import google.generativeai as genai
from mesa import Agent, Model

# Goal: Compress older memories into a single token-efficient summary to prevent token exhaustion.

class SummarizingAgent(Agent):
    def __init__(self, model):
        super().__init__(model) 
        self.raw_memory = []
        self.compressed_summary = "No past memory."
        
        genai.configure(api_key=os.environ.get("GEMINI_API_KEY"))
        self.llm = genai.GenerativeModel('gemini-2.5-flash')

    def step(self):
        print(f"\n--- Agent {self.unique_id} Step {self.model.steps} ---")
        
        # 1. Simulate observing a new event
        new_event = f"Step {self.model.steps}: Explored a new sector and gathered data."
        self.raw_memory.append(new_event)
        print(f"Observed: {new_event}")

        # 2. Trigger summarization if memory array gets too long
        if len(self.raw_memory) > 3:
            print(f"[SYSTEM] Token limit approaching. Triggering LLM Memory Compression...")
            self.compress_memories()

        print(f"Current Working Context: {self.compressed_summary}")

    def compress_memories(self):
        prompt = f"Summarize these consecutive events into a single, highly concise sentence: {self.raw_memory}"
        try:
            response = self.llm.generate_content(prompt)
            self.compressed_summary = response.text.strip()
            self.raw_memory = [] # Flush raw memory after successful compression
            print(f"[SUCCESS] Memory compressed to save tokens.")
        except Exception as e:
            print(f"[ERROR] Summarization bypassed due to API load/error: {e}")

class SummarizationModel(Model):
    def __init__(self):
        super().__init__()
        self.test_agent = SummarizingAgent(self)

    def step(self):

        self.test_agent.step()

In [6]:
if __name__ == "__main__":
    print("Initiating Phase 11: Memory Summarization Engine...")
    model = SummarizationModel()
    for i in range(5):
        model.step()

Initiating Phase 11: Memory Summarization Engine...

--- Agent 1 Step 1 ---
Observed: Step 1: Explored a new sector and gathered data.
Current Working Context: No past memory.

--- Agent 1 Step 2 ---
Observed: Step 2: Explored a new sector and gathered data.
Current Working Context: No past memory.

--- Agent 1 Step 3 ---
Observed: Step 3: Explored a new sector and gathered data.
Current Working Context: No past memory.

--- Agent 1 Step 4 ---
Observed: Step 4: Explored a new sector and gathered data.
[SYSTEM] Token limit approaching. Triggering LLM Memory Compression...
[SUCCESS] Memory compressed to save tokens.
Current Working Context: Explored multiple new sectors and gathered data.

--- Agent 1 Step 5 ---
Observed: Step 5: Explored a new sector and gathered data.
Current Working Context: Explored multiple new sectors and gathered data.
